In [ ]:
%pip install -q captum

In [ ]:
from pathlib import Path
import torch
from captum.attr import IntegratedGradients

root = Path("/kaggle/input")

metadata_files = list(root.rglob("metadata.npz"))
model_files = list(root.rglob("best_model.pt"))

assert len(metadata_files) == 1, "请检查 ECG-Insight-01 输入"
assert len(model_files) == 1, "请只添加 ResNet 模型输入"

prepared_dir = metadata_files[0].parent
model_path = model_files[0]

assert (prepared_dir / "X.npy").is_file()
assert (model_path.parent / "thresholds.npy").is_file()

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Captum 导入成功")
print("训练设备：", device)
if device.type == "cuda":
    print("GPU：", torch.cuda.get_device_name(0))

print("数据目录：", prepared_dir)
print("模型文件：", model_path)
print("环境与文件检查通过")

In [ ]:
import numpy as np
import pandas as pd
from torch import nn

# 结构必须与训练时一致
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.main = nn.Sequential(
            nn.Conv1d(
                in_channels, out_channels,
                kernel_size=5, stride=stride,
                padding=2, bias=False,
            ),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(),
            nn.Conv1d(
                out_channels, out_channels,
                kernel_size=5, padding=2, bias=False,
            ),
            nn.BatchNorm1d(out_channels),
        )

        if in_channels != out_channels or stride != 1:
            self.shortcut = nn.Sequential(
                nn.Conv1d(
                    in_channels, out_channels,
                    kernel_size=1, stride=stride, bias=False,
                ),
                nn.BatchNorm1d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

        self.activation = nn.ReLU()

    def forward(self, x):
        return self.activation(self.main(x) + self.shortcut(x))

class ECGResNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(12, 32, kernel_size=7, padding=3, bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            ResidualBlock(32, 32),
            ResidualBlock(32, 64, stride=2),
            ResidualBlock(64, 128, stride=2),
            nn.AdaptiveAvgPool1d(1),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.2),
            nn.Linear(128, 5),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# 加载已保存的权重，不训练
model = ECGResNet().to(device)
checkpoint = torch.load(
    model_path,
    map_location=device,
    weights_only=True,
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

info = np.load(prepared_dir / "metadata.npz")
X = np.load(prepared_dir / "X.npy", mmap_mode="r")
classes = info["classes"]

assert checkpoint["classes"] == classes.tolist()

ecg_id = 1745
positions = np.flatnonzero(info["ecg_ids"] == ecg_id)
assert len(positions) == 1
position = positions[0]

sample_tensor = torch.from_numpy(
    np.array(X[position], dtype=np.float32, copy=True)
).unsqueeze(0).to(device)

with torch.no_grad():
    probabilities = torch.sigmoid(model(sample_tensor)).cpu().numpy()[0]

# 与保存的测试预测逐类比较
saved = np.load(model_path.parent / "test_predictions.npz")
saved_positions = np.flatnonzero(saved["ecg_ids"] == ecg_id)
assert len(saved_positions) == 1

expected = saved["probabilities"][saved_positions[0]]
np.testing.assert_allclose(
    probabilities, expected, rtol=1e-4, atol=1e-5
)

display(pd.DataFrame({
    "类别": classes,
    "恢复模型预测": probabilities,
    "原保存预测": expected,
}).round(4))

print("核对通过：恢复模型与原保存预测一致。")

In [ ]:
from captum.attr import IntegratedGradients
from pathlib import Path

target_index = list(classes).index("HYP")
baseline = torch.zeros_like(sample_tensor)

ig = IntegratedGradients(model)

# 分批计算积分路径，控制 GPU 内存
attributions, delta = ig.attribute(
    sample_tensor,
    baselines=baseline,
    target=target_index,
    n_steps=128,
    internal_batch_size=16,
    return_convergence_delta=True,
)

with torch.no_grad():
    input_logit = model(sample_tensor)[0, target_index].item()
    baseline_logit = model(baseline)[0, target_index].item()

attr = attributions.detach().cpu().numpy()[0]
logit_difference = input_logit - baseline_logit
attr_sum = float(attr.sum())
absolute_error = abs(float(delta.item()))
relative_error = absolute_error / max(abs(logit_difference), 1e-8)

assert np.isfinite(attr).all()

print("目标类别：HYP")
print(f"样本 logit：{input_logit:.6f}")
print(f"参考输入 logit：{baseline_logit:.6f}")
print(f"logit 差值：{logit_difference:.6f}")
print(f"归因总和：{attr_sum:.6f}")
print(f"完整性残差绝对值：{absolute_error:.6f}")
print(f"相对残差：{relative_error:.2%}")

# 按导联汇总：净归因与归因绝对值总量
lead_summary = pd.DataFrame({
    "导联": info["leads"],
    "净归因": attr.sum(axis=1),
    "归因绝对值总量": np.abs(attr).sum(axis=1),
}).sort_values("归因绝对值总量", ascending=False)

display(lead_summary.round(4))

interpret_dir = Path("/kaggle/working/interpretability")
interpret_dir.mkdir(exist_ok=True)

np.savez(
    interpret_dir / "ecg_1745_hyp_ig.npz",
    attributions=attr,
    ecg_id=ecg_id,
    target="HYP",
    baseline="zero_in_standardized_space",
    n_steps=128,
    input_logit=input_logit,
    baseline_logit=baseline_logit,
    convergence_delta=float(delta.item()),
)
lead_summary.to_csv(
    interpret_dir / "ecg_1745_hyp_lead_summary.csv",
    index=False,
)

In [ ]:
import matplotlib.pyplot as plt

# 还原原始电压尺度
waveform = (
    np.asarray(X[position], dtype=np.float32)
    * info["std"][0]
    + info["mean"][0]
)
time = np.arange(waveform.shape[1]) / 100

# 使用统一对称色标，保留正负方向
color_limit = max(float(np.abs(attr).max()), 1e-12)

fig, axes = plt.subplots(
    2, 1,
    figsize=(16, 9),
    constrained_layout=True,
)

# 上图：每条导联沿纵轴错开放置，所有导联使用相同电压比例
offset = max(float(np.ptp(waveform, axis=1).max()) * 1.2, 1.0)

lead_positions = np.arange(len(info["leads"]))[::-1] * offset

for i, lead in enumerate(info["leads"]):
    axes[0].plot(
        time,
        waveform[i] + lead_positions[i],
        linewidth=0.7,
    )

axes[0].set_yticks(
    lead_positions[::-1],
    labels=info["leads"][::-1],
)
axes[0].set_xlim(0, 10)
axes[0].set_title("ECG 1745 | Waveforms with vertical offsets")
axes[0].set_xlabel("Time (s)")
axes[0].grid(alpha=0.2)

# 下图：红色推动 HYP logit，蓝色抑制 HYP logit
heatmap = axes[1].imshow(
    attr,
    aspect="auto",
    cmap="RdBu_r",
    vmin=-color_limit,
    vmax=color_limit,
    extent=[0, 10, 11.5, -0.5],
    interpolation="nearest",
)

axes[1].set_yticks(
    np.arange(len(info["leads"])),
    labels=info["leads"],
)
axes[1].set_xlabel("Time (s)")
axes[1].set_title(
    "Integrated Gradients for HYP logit | "
    "Red: positive, Blue: negative"
)
fig.colorbar(
    heatmap,
    ax=axes[1],
    label="Signed attribution",
)

fig.savefig(
    interpret_dir / "ecg_1745_hyp_ig_heatmap.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

In [ ]:
# 从已保存的测试预测中选择 HYP 真阳性
hyp_index = list(classes).index("HYP")
hyp_scores = saved["probabilities"][:, hyp_index]
hyp_labels = saved["targets"][:, hyp_index]
hyp_threshold = np.load(model_path.parent / "thresholds.npy")[hyp_index]

tp_indices = np.flatnonzero(
    (hyp_labels == 1) & (hyp_scores >= hyp_threshold)
)
assert len(tp_indices) > 0, "没有找到 HYP 真阳性"

# 按预测分数排序，选择中间位置的记录
ordered = tp_indices[np.argsort(hyp_scores[tp_indices])]
selected = ordered[len(ordered) // 2]
tp_ecg_id = int(saved["ecg_ids"][selected])

print("HYP 真阳性数量：", len(tp_indices))
print("对照记录编号：", tp_ecg_id)
print("HYP 预测分数：", round(float(hyp_scores[selected]), 4))
print("HYP 固定阈值：", round(float(hyp_threshold), 4))

display(pd.DataFrame({
    "类别": classes,
    "真实标签": saved["targets"][selected].astype(int),
    "预测分数": saved["probabilities"][selected],
}).round(4))

In [ ]:
tp_positions = np.flatnonzero(info["ecg_ids"] == tp_ecg_id)
assert len(tp_positions) == 1
tp_position = tp_positions[0]

tp_tensor = torch.from_numpy(
    np.array(X[tp_position], dtype=np.float32, copy=True)
).unsqueeze(0).to(device)

tp_baseline = torch.zeros_like(tp_tensor)

# 核对恢复模型的预测
with torch.no_grad():
    tp_logits = model(tp_tensor)
    tp_prob = torch.sigmoid(tp_logits).cpu().numpy()[0]
    tp_baseline_logit = model(tp_baseline)[0, hyp_index].item()

np.testing.assert_allclose(
    tp_prob,
    saved["probabilities"][selected],
    rtol=1e-4,
    atol=1e-5,
)

# 与漏报样本使用相同设置
tp_attributions, tp_delta = ig.attribute(
    tp_tensor,
    baselines=tp_baseline,
    target=hyp_index,
    n_steps=128,
    internal_batch_size=16,
    return_convergence_delta=True,
)

tp_attr = tp_attributions.detach().cpu().numpy()[0]
assert np.isfinite(tp_attr).all()

tp_input_logit = tp_logits[0, hyp_index].item()
tp_difference = tp_input_logit - tp_baseline_logit
tp_error = abs(float(tp_delta.item()))
tp_relative_error = tp_error / max(abs(tp_difference), 1e-8)

print("记录编号：", tp_ecg_id)
print(f"HYP 预测分数：{tp_prob[hyp_index]:.4f}")
print(f"logit 差值：{tp_difference:.6f}")
print(f"归因总和：{tp_attr.sum():.6f}")
print(f"完整性残差绝对值：{tp_error:.6f}")
print(f"相对残差：{tp_relative_error:.2%}")

tp_lead_summary = pd.DataFrame({
    "导联": info["leads"],
    "净归因": tp_attr.sum(axis=1),
    "归因绝对值总量": np.abs(tp_attr).sum(axis=1),
}).sort_values("归因绝对值总量", ascending=False)

display(tp_lead_summary.round(4))

np.savez(
    interpret_dir / f"ecg_{tp_ecg_id}_hyp_ig.npz",
    attributions=tp_attr,
    ecg_id=tp_ecg_id,
    target="HYP",
    baseline="zero_in_standardized_space",
    n_steps=128,
    input_logit=tp_input_logit,
    baseline_logit=tp_baseline_logit,
    convergence_delta=float(tp_delta.item()),
)

tp_lead_summary.to_csv(
    interpret_dir / f"ecg_{tp_ecg_id}_hyp_lead_summary.csv",
    index=False,
)

In [ ]:
import matplotlib.pyplot as plt

fn_waveform = (
    np.asarray(X[position], dtype=np.float32)
    * info["std"][0] + info["mean"][0]
)
tp_waveform = (
    np.asarray(X[tp_position], dtype=np.float32)
    * info["std"][0] + info["mean"][0]
)

# 两个案例使用统一尺度
shared_limit = max(
    float(np.abs(attr).max()),
    float(np.abs(tp_attr).max()),
    1e-12,
)

offset = max(
    float(np.ptp(fn_waveform, axis=1).max()),
    float(np.ptp(tp_waveform, axis=1).max()),
    1.0,
) * 1.2

lead_positions = np.arange(12)[::-1] * offset
time = np.arange(1000) / 100

fig, axes = plt.subplots(
    2, 2,
    figsize=(18, 10),
    constrained_layout=True,
)

examples = [
    (1745, "False negative", fn_waveform, attr),
    (tp_ecg_id, "True positive", tp_waveform, tp_attr),
]

for col, (record_id, status, wave, attribution) in enumerate(examples):
    for i in range(12):
        axes[0, col].plot(
            time, wave[i] + lead_positions[i],
            linewidth=0.7,
        )

    axes[0, col].set_yticks(
        lead_positions[::-1],
        labels=info["leads"][::-1],
    )
    axes[0, col].set_xlim(0, 10)
    axes[0, col].set_title(f"ECG {record_id} | {status}")
    axes[0, col].set_xlabel("Time (s)")
    axes[0, col].grid(alpha=0.2)

    heatmap = axes[1, col].imshow(
        attribution,
        aspect="auto",
        cmap="RdBu_r",
        vmin=-shared_limit,
        vmax=shared_limit,
        extent=[0, 10, 11.5, -0.5],
        interpolation="nearest",
    )

    axes[1, col].set_yticks(range(12), labels=info["leads"])
    axes[1, col].set_xlabel("Time (s)")
    axes[1, col].set_title("HYP logit attribution")

# 上排统一纵轴范围
lower = min(ax.get_ylim()[0] for ax in axes[0])
upper = max(ax.get_ylim()[1] for ax in axes[0])
for ax in axes[0]:
    ax.set_ylim(lower, upper)

fig.colorbar(
    heatmap,
    ax=axes[1, :].tolist(),
    label="Signed attribution | Red: positive, Blue: negative",
)

fig.savefig(
    interpret_dir / "hyp_fn_tp_comparison.png",
    dpi=180,
    bbox_inches="tight",
)
plt.show()

In [ ]:
report = """# HYP 积分梯度案例对照

## 方法
使用已保存的 ECGResNet，解释 HYP 原始输出 logit。
参考输入为标准化空间的全零数组，对应训练集各导联
平均电压，不代表真实正常心电图。
积分步数为 128，两个案例使用统一热图色标。

## 案例与结果
- 漏报记录 1745：HYP 分数约 0.0062。
  相对于参考输入，logit 差值约 -2.867；
  积分完整性相对残差约 0.14%。
- 真阳性记录 1320：HYP 分数约 0.6579。
  同时包含 MI、STTC 标签。
  logit 差值约 +2.861，相对残差约 0.32%。
  V6 的净正归因及归因绝对值总量较突出。

## 图像观察
两个案例的归因均在部分重复波形区域出现。
记录 1320 的 V6 可见缓慢基线起伏，并有归因分布，
需要进一步检查信号质量及模型对该变化的敏感性。

## 限制
小残差只说明数值积分近似满足完整性，
不证明医学解释正确。
归因依赖参考输入，未开展参考输入敏感性分析。
两个样本标签组合不同，不能单独解释漏报原因，
也不能代表全部 HYP 样本。
热图不作为病灶定位或临床诊断依据。
"""

(interpret_dir / "hyp_ig_comparison_report.md").write_text(
    report, encoding="utf-8"
)

print("归因对照报告已保存")

In [ ]:
from scipy.signal import butter, sosfiltfilt

# 只改变 V6，其他导联保持原样
v6_index = list(info["leads"]).index("V6")

# 将标准化数据还原为电压
original_wave = (
    np.asarray(X[tp_position], dtype=np.float32)
    * info["std"][0] + info["mean"][0]
)

# 0.5 Hz 高通：作为探索性扰动，不作为新的预处理方案
sos = butter(
    2, 0.5, btype="highpass", fs=100, output="sos"
)

modified_wave = original_wave.copy()
modified_wave[v6_index] = sosfiltfilt(
    sos, original_wave[v6_index]
)

# 使用原训练集参数重新标准化
modified_input = (
    (modified_wave - info["mean"][0]) / info["std"][0]
).astype(np.float32)

modified_tensor = torch.from_numpy(
    modified_input.copy()
).unsqueeze(0).to(device)

with torch.no_grad():
    original_logits = model(tp_tensor)[0].cpu().numpy()
    modified_logits = model(modified_tensor)[0].cpu().numpy()
    original_probs = torch.sigmoid(model(tp_tensor))[0].cpu().numpy()
    modified_probs = torch.sigmoid(model(modified_tensor))[0].cpu().numpy()

sensitivity = pd.DataFrame({
    "类别": classes,
    "原预测分数": original_probs,
    "仅V6高通后分数": modified_probs,
    "分数变化": modified_probs - original_probs,
    "logit变化": modified_logits - original_logits,
})
display(sensitivity.round(4))

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(time, original_wave[v6_index], label="Original V6")
ax.plot(time, modified_wave[v6_index], label="High-pass V6", alpha=0.8)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude (mV)")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

fig.savefig(
    interpret_dir / "ecg_1320_v6_sensitivity.png",
    dpi=150,
)
sensitivity.to_csv(
    interpret_dir / "ecg_1320_v6_sensitivity.csv",
    index=False,
)
plt.show()

In [ ]:
report_path = interpret_dir / "hyp_ig_comparison_report.md"

section = """
## V6 单导联敏感性检查
对记录 1320 的 V6 应用二阶 0.5 Hz 高通。
其他导联、模型和分类阈值保持不变，
并使用原训练集参数重新标准化。

HYP 分数从约 0.6579 变为 0.6754，
变化约 +0.0175，仍高于固定阈值 0.3261。
本次扰动下，HYP 阳性判断保持不变。

结果未支持“V6 缓慢基线起伏导致阳性预测”的猜测。
但高通并非只删除噪声，此单样本实验不能证明
模型对基线变化普遍稳健。
"""

text = report_path.read_text(encoding="utf-8")

if "## V6 单导联敏感性检查" not in text:
    report_path.write_text(text + section, encoding="utf-8")

print("敏感性检查结果已追加到报告")

In [15]:
import zipfile

# 已分析的漏报、误报与真阳性，共 7 条
demo_ids = np.array(
    [1745, 1320, 3215, 1556, 20224, 16920, 9167]
)

demo_positions = []
prediction_positions = []

for record_id in demo_ids:
    data_match = np.flatnonzero(info["ecg_ids"] == record_id)
    pred_match = np.flatnonzero(saved["ecg_ids"] == record_id)

    assert len(data_match) == len(pred_match) == 1
    demo_positions.append(data_match[0])
    prediction_positions.append(pred_match[0])

demo_positions = np.array(demo_positions)
prediction_positions = np.array(prediction_positions)

# 保存标准化波形及还原电压所需的参数
demo_dir = Path("/kaggle/working/demo_bundle")
demo_dir.mkdir(exist_ok=True)

np.savez_compressed(
    demo_dir / "demo_samples.npz",
    X=np.asarray(X[demo_positions], dtype=np.float32),
    ecg_ids=demo_ids,
    targets=saved["targets"][prediction_positions],
    probabilities=saved["probabilities"][prediction_positions],
    thresholds=saved["thresholds"],
    classes=info["classes"],
    leads=info["leads"],
    mean=info["mean"],
    std=info["std"],
    sampling_rate=100,
)

# 打包演示样本及已生成的归因成果
zip_path = Path("/kaggle/working/ecg_demo_bundle.zip")

with zipfile.ZipFile(
    zip_path, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    archive.write(
        demo_dir / "demo_samples.npz",
        arcname="data/demo_samples.npz",
    )

    for file in sorted(interpret_dir.iterdir()):
        if file.is_file():
            archive.write(
                file,
                arcname=f"interpretability/{file.name}",
            )

print("演示样本数：", len(demo_ids))
print("打包文件：", zip_path)
print(f"大小：{zip_path.stat().st_size / 1024**2:.2f} MiB")

演示样本数： 7
打包文件： /kaggle/working/ecg_demo_bundle.zip
大小：1.01 MiB
